# Task 5: Deep Learning / NLP Text Classifier with PyTorch

## Objective
Build an NLP text classification pipeline using TF-IDF vectorization and a multi-layer PyTorch neural network with dropout, batch normalization, and early stopping. Train and validate the model, plot loss/accuracy convergence curves, and test unseen text inputs with classification confidence.

**Dataset:** A binary subset of the public 20 Newsgroups text dataset. Two categories are selected to keep the demonstration focused and reproducible.


In [ ]:
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1. Load and prepare text data

In [ ]:
categories = ['sci.space', 'rec.sport.baseball']

train_raw = fetch_20newsgroups(
    subset='train', categories=categories,
    remove=('headers', 'footers', 'quotes'), random_state=SEED
)

test_raw = fetch_20newsgroups(
    subset='test', categories=categories,
    remove=('headers', 'footers', 'quotes'), random_state=SEED
)

texts = np.array(train_raw.data)
labels = np.array(train_raw.target)
test_texts = np.array(test_raw.data)
test_labels = np.array(test_raw.target)

print('Classes:', train_raw.target_names)
print('Training documents:', len(texts))
print('Test documents:', len(test_texts))

## 2. Train/validation split

The vectorizer is fitted only on the training split. This avoids using validation vocabulary statistics during training.

In [ ]:
X_text_train, X_text_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.20, random_state=SEED, stratify=labels
)

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    stop_words='english'
)

X_train = vectorizer.fit_transform(X_text_train).astype(np.float32).toarray()
X_val = vectorizer.transform(X_text_val).astype(np.float32).toarray()
X_test = vectorizer.transform(test_texts).astype(np.float32).toarray()

print('TF-IDF feature count:', X_train.shape[1])

## 3. PyTorch datasets and data loaders

In [ ]:
train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train, dtype=torch.long))
val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val, dtype=torch.long))
test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(test_labels, dtype=torch.long))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

## 4. Multi-layer neural network

The model contains multiple dense layers, BatchNorm, ReLU activations, and Dropout.

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.40),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)

model = TextClassifier(X_train.shape[1]).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
print(model)

## 5. Training with validation and early stopping

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        if training:
            optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        if training:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += len(yb)

    return total_loss / total, correct / total

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_state = None
best_val_loss = float('inf')
patience = 4
wait = 0
epochs = 20

for epoch in range(1, epochs + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    with torch.no_grad():
        val_loss, val_acc = run_epoch(model, val_loader, criterion)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch:02d} | train loss {train_loss:.4f} | val loss {val_loss:.4f} | train acc {train_acc:.4f} | val acc {val_acc:.4f}')

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print('Early stopping triggered.')
            break

model.load_state_dict(best_state)

## 6. Loss and accuracy convergence curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

plt.figure(figsize=(9, 5))
plt.plot(epochs_ran, history['train_loss'], label='Train loss')
plt.plot(epochs_ran, history['val_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(epochs_ran, history['train_acc'], label='Train accuracy')
plt.plot(epochs_ran, history['val_acc'], label='Validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Evaluate on unseen test documents

In [ ]:
model.eval()
all_preds, all_probs = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb.to(DEVICE))
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_preds.append(probs.argmax(axis=1))

test_probs = np.vstack(all_probs)
test_preds = np.concatenate(all_preds)

print('Accuracy:', accuracy_score(test_labels, test_preds))
print('Precision:', precision_score(test_labels, test_preds, zero_division=0))
print('Recall:', recall_score(test_labels, test_preds, zero_division=0))
print('F1:', f1_score(test_labels, test_preds, zero_division=0))
print('\nClassification report:')
print(classification_report(test_labels, test_preds, target_names=train_raw.target_names, zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(test_labels, test_preds))

## 8. Unseen sample inference with confidence

In [ ]:
sample_texts = [
    'The spacecraft entered orbit around the planet and scientists are analyzing the mission data.',
    'The pitcher struck out the final batter and the team won the baseball game.',
    'Researchers are planning a new space mission and testing a satellite communication system.'
]

sample_vectors = vectorizer.transform(sample_texts).astype(np.float32).toarray()
sample_tensor = torch.tensor(sample_vectors).to(DEVICE)

model.eval()
with torch.no_grad():
    sample_probs = torch.softmax(model(sample_tensor), dim=1).cpu().numpy()

for text, probs in zip(sample_texts, sample_probs):
    idx = int(np.argmax(probs))
    print('\nText:', text)
    print('Predicted class:', train_raw.target_names[idx])
    print('Confidence:', f'{probs[idx] * 100:.2f}%')
    print('Class probabilities:', dict(zip(train_raw.target_names, np.round(probs, 4))))

## 9. Save the trained model and vectorizer

Both the PyTorch model weights and TF-IDF vectorizer are saved so that the same preprocessing can be reused during inference.

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'input_dim': X_train.shape[1],
    'class_names': train_raw.target_names
}, 'text_classifier_pytorch.pt')

import joblib
joblib.dump(vectorizer, 'tfidf_vectorizer.joblib')

print('Saved text_classifier_pytorch.pt and tfidf_vectorizer.joblib')

## Final checklist

- [x] Text tokenization/vectorization with TF-IDF
- [x] Multi-layer neural network
- [x] Dropout
- [x] Batch normalization
- [x] Early stopping callback logic
- [x] Training and validation loss curves
- [x] Training and validation accuracy curves
- [x] Evaluation on unseen test documents
- [x] Classification confidence outputs
- [x] Saved model and vectorizer artifacts
